In [50]:
import pandas as pd
import requests
import json
from sklearn.linear_model import LinearRegression
from collections import defaultdict

https://pantherdb.org/services/openAPISpec.jsp panther api doc

In [2]:
df15_r = (
    pd.read_csv('correlations/GEP_15_analyzed.csv')
    .drop('Unnamed: 0', axis=1).set_index('gene')
    .rename(columns={'GEP 15 (DNA Replication)':'GEP 15'})
)
df22_r = (
    pd.read_csv('correlations/GEP_22_analyzed.csv')
    .drop('Unnamed: 0', axis=1).set_index('gene')
    .rename(columns={'GEP 22 (Cell cycle)': 'GEP 22'})
)

In [3]:
hsc_15_lr = LinearRegression().fit(df15_r[['r hsc']], df15_r['GEP 15'])
prog_15_lr = LinearRegression().fit(df15_r[['r prog']], df15_r['GEP 15'])
hsc_22_lr = LinearRegression().fit(df22_r[['r hsc']], df22_r['GEP 22'])
prog_22_lr = LinearRegression().fit(df22_r[['r prog']], df22_r['GEP 22'])

In [4]:
def pred_GEP_expression(df, model, cell_type, GEP):
    assert cell_type in ['hsc', 'prog']
    assert GEP in [15, 22]
    
    return df.assign(**{f'GEP {GEP} pred based on r {cell_type}': model.predict(df[[f'r {cell_type}']])})

In [5]:
df15 = (df15_r
        .pipe(pred_GEP_expression, hsc_15_lr, 'hsc', 15)
        .pipe(pred_GEP_expression, prog_15_lr, 'prog', 15)
       )
df22 = (df22_r
        .pipe(pred_GEP_expression, hsc_22_lr, 'hsc', 22)
        .pipe(pred_GEP_expression, prog_22_lr, 'prog', 22)
       )

In [6]:
def get_panther_lists(df, GEP):
    # returns tuple of DFs to put into panther
    # filter only for regression significantly overshooting/undershooting
    # (HSC overshoot, HSC undershoot, prog overshoot, prog undershoot)
    
    assert GEP in [15, 22]
    p_cutoff = 0.05
    
    df_hsc = df[df[f'adj regression p of GEP {GEP} vs r hsc'] < p_cutoff]
    df_hsc_over = df_hsc[df_hsc[f'GEP {GEP} pred based on r hsc'] > df_hsc[f'GEP {GEP}']]
    df_hsc_under = df_hsc[df_hsc[f'GEP {GEP} pred based on r hsc'] < df_hsc[f'GEP {GEP}']]
    
    df_prog = df[df[f'adj regression p of GEP {GEP} vs r prog'] < p_cutoff]
    df_prog_over = df_hsc[df_hsc[f'GEP {GEP} pred based on r prog'] > df_hsc[f'GEP {GEP}']]
    df_prog_under = df_hsc[df_hsc[f'GEP {GEP} pred based on r prog'] < df_hsc[f'GEP {GEP}']]
    
    return df_hsc_over, df_hsc_under, df_prog_over, df_prog_under

In [7]:
df15_hsc_over, df15_hsc_under, df15_prog_over, df15_prog_under = df15.pipe(get_panther_lists, 15)
df22_hsc_over, df22_hsc_under, df22_prog_over, df22_prog_under = df22.pipe(get_panther_lists, 22)

In [8]:
def get_panther_list(df):
    return ','.join(df.index.to_list())

In [81]:
dfs = [df15_hsc_over, df15_hsc_under, df15_prog_over, df15_prog_under, df22_hsc_over, df22_hsc_under, df22_prog_over, df22_prog_under]

In [84]:
def get_panther_dct(df):
    lst = df.pipe(get_panther_list)
    
    organism = "9606" # humans
    dataset = "GO:0003674" # molecular function
    geneinfo_url = "https://pantherdb.org/services/oai/pantherdb/enrich/overrep"

    params = {
        "geneInputList": lst,
        "organism": organism,
        "annotDataSet": dataset,
        "mappedInfo": 'COMP_LIST'
    }
    response = requests.post(geneinfo_url, data=params)

    data = response.json()
    
    res = defaultdict(list)
    for d in data['results']['result']:
        if d['number_in_list'] > 0 and d['fdr'] < 1:
            k = d['input_list']['mapped_ids']
            res[k].append(d['term']['label'])

In [ ]:
get_panther_dct(df15_hsc_over)